In [1]:
import os
import math
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from collections import Counter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
documents = [
    "Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니다.",
    "자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.",
    "벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하는 시스템입니다.",
    "GPT-4는 OpenAI가 개발한 대규모 언어 모델로 다양한 작업을 수행합니다.",
    "RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM 응답을 개선합니다.",
    "FastAPI는 Python으로 빠른 웹 API를 구축하기 위한 프레임워크입니다.",
    "트랜스포머 아키텍처는 어텐션 메커니즘을 활용한 딥러닝 모델 구조입니다.",
    "FAISS는 Facebook AI가 개발한 효율적인 유사도 검색 라이브러리입니다.",
    "프롬프트 엔지니어링은 LLM에 효과적인 입력을 설계하는 기술입니다.",
    "임베딩은 텍스트를 수치 벡터로 변환하여 의미적 유사성을 측정할 수 있게 합니다.",
]

In [2]:
doc_embeddings = embeddings.embed_documents(documents)
len(doc_embeddings), len(doc_embeddings[0])

(10, 1536)

In [3]:
!uv add sentence-transformers

Resolved 223 packages in 901ms                                       
Prepared 9 packages in 14.75s                                            
Uninstalled 1 package in 77ms
Installed 9 packages in 353ms                               
 + mpmath==1.3.0
 + networkx==3.4.2
 + safetensors==0.7.0
 + sentence-transformers==5.3.0
 - setuptools==82.0.1
 + setuptools==81.0.0
 + sympy==1.14.0
 + tokenizers==0.22.2
 + torch==2.11.0
 + transformers==5.4.0


In [5]:
from sentence_transformers import SentenceTransformer

In [6]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embedding = embedding_model.encode(documents)

Loading weights: 100%|██████████████████████| 103/103 [00:00<00:00, 9421.09it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
embedding.shape

(10, 384)

In [8]:
embedding

array([[-0.02818311,  0.02315178, -0.01332183, ...,  0.09440263,
         0.06264345, -0.00686624],
       [ 0.004331  ,  0.02231258,  0.06852962, ...,  0.06287047,
         0.03236052,  0.01451776],
       [ 0.04224597,  0.06530675,  0.02390072, ...,  0.01747432,
        -0.08692615,  0.02379496],
       ...,
       [-0.03313584,  0.02055405,  0.00575812, ...,  0.07556442,
        -0.08282115, -0.06177336],
       [ 0.09051041, -0.00739943,  0.0402192 , ..., -0.00768869,
        -0.11420641, -0.01269386],
       [ 0.01094474,  0.08185513,  0.03489934, ...,  0.01663285,
        -0.08264508,  0.02735938]], shape=(10, 384), dtype=float32)

In [9]:
def keyword_search(query, docs, top_k=3):
    query_tokens = set(query.lower().split())
    scores = []
    for i, doc in enumerate(docs):
        doc_tokens = set(doc.lower().split())
        overlap = len(query_tokens & doc_tokens)
        scores.append((i, overlap))
    
    scores.sort(key=lambda x:x[1], reverse=True)
    return scores[:top_k]

In [11]:
results = keyword_search("Python 프로그래밍 언어", documents)
results

for idx, scores in results:
    print(f"[{idx}] overlap = {scores} | {documents[idx][:30]}")

[0] overlap = 1 | Python은 데이터 과학과 머신러닝에 널리 사용되는 
[3] overlap = 1 | GPT-4는 OpenAI가 개발한 대규모 언어 모델로 
[1] overlap = 0 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고


In [13]:
def vector_serach(query, docs, doc_embs, top_k=3):
    q_emb = np.array(embeddings.embed_query(query))
    # 코사인 유사도에 대한 값이 scores 1에 가까울수록 좋음
    similarities = np.dot(doc_embs, q_emb) / (np.linalg.norm(doc_embs, axis=1) * np.linalg.norm(q_emb))
    top_indices = similarities.argsort()[::-1][:top_k]
    return [(i, similarities[i]) for i in top_indices]

results = vector_serach("Python 프로그래밍 언어", documents, np.array(doc_embeddings), top_k = 3)

In [15]:
for idx, scores in results:
    print(f"[{idx}] similarity = {scores} | {documents[idx][:30]}")

[0] similarity = 0.5528885319764565 | Python은 데이터 과학과 머신러닝에 널리 사용되는 
[1] similarity = 0.37176929945476994 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고
[3] similarity = 0.36039831493805485 | GPT-4는 OpenAI가 개발한 대규모 언어 모델로 


In [16]:
results = vector_serach('FAISS', documents, np.array(doc_embeddings), top_k = 3)

In [17]:
for idx, scores in results:
    print(f"[{idx}] similarity = {scores} | {documents[idx][:30]}")

[7] similarity = 0.5278809597501696 | FAISS는 Facebook AI가 개발한 효율적인 유
[4] similarity = 0.19233440678444033 | RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM
[5] similarity = 0.09642334637561256 | FastAPI는 Python으로 빠른 웹 API를 구축


In [18]:
def overlap_rate(keyword_results, vector_results):
    kw_ids = set(idx for idx, _ in keyword_results)
    vec_ids = set(idx for idx, _ in vector_results)

    overlap = kw_ids & vec_ids
    union = kw_ids | vec_ids

    return len(overlap) / len(union)

In [20]:
for query in ['Python 프로그래밍 언어', '딥러닝 모델', 'FAISS 라이브러리']:
    kw = keyword_search(query, documents, top_k = 5)
    vec = vector_serach(query, documents, np.array(doc_embeddings), top_k = 5)
    rate = overlap_rate(kw, vec)
    print(f"{query} overlap : {rate}")

Python 프로그래밍 언어 overlap : 0.42857142857142855
딥러닝 모델 overlap : 0.42857142857142855
FAISS 라이브러리 overlap : 0.25


In [22]:
def simple_hybrid(query, docs, doc_embs, top_k=3):
    kw = keyword_search(query, documents, top_k = len(docs))
    vec = vector_serach(query, documents, np.array(doc_embs), top_k= len(docs))

    kw_scores = {idx : score for idx, score in kw}
    vec_scores = {idx : score for idx, score in vec}

    kw_max = max(kw_scores.values()) or 1
    vec_max = max(vec_scores.values()) or 1

    combined = {}

    for idx in range(len(docs)):
        kw_score = kw_scores.get(idx, 0)/ kw_max
        vec_score = vec_scores.get(idx, 0)/ vec_max

        combined[idx] = kw_score + vec_score

    ranked = sorted(combined.items(), key=lambda x:x[1], reverse=True)
    return ranked[:top_k]

In [25]:
for query in ['Python 프로그래밍 언어', '딥러닝 모델 구조', 'FAISS 라이브러리']:
    results = simple_hybrid(query, documents, np.array(doc_embeddings), top_k = 3)

    print(results)
    for idx, score in results:
        print(f"[{idx}] {score} | {documents[idx][:30]}")

[(0, np.float64(2.0)), (3, np.float64(1.6519851524685754)), (1, np.float64(0.672504502690985))]
[0] 2.0 | Python은 데이터 과학과 머신러닝에 널리 사용되는 
[3] 1.6519851524685754 | GPT-4는 OpenAI가 개발한 대규모 언어 모델로 
[1] 0.672504502690985 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고
[(6, np.float64(2.0)), (3, np.float64(0.49054669263638906)), (0, np.float64(0.4189813664317752))]
[6] 2.0 | 트랜스포머 아키텍처는 어텐션 메커니즘을 활용한 딥러닝 
[3] 0.49054669263638906 | GPT-4는 OpenAI가 개발한 대규모 언어 모델로 
[0] 0.4189813664317752 | Python은 데이터 과학과 머신러닝에 널리 사용되는 
[(7, np.float64(1.0)), (4, np.float64(0.37548873259884774)), (2, np.float64(0.3276636725616183))]
[7] 1.0 | FAISS는 Facebook AI가 개발한 효율적인 유
[4] 0.37548873259884774 | RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM
[2] 0.3276636725616183 | 벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 


In [27]:
"""
TF-IDF : 단어 중요도를 숫자로 표현

TF(Term Frequency) : 용어 빈도
f(t, d) / | d | document에 나온 t의 개수, |d| : d의 길이
해당 문서에서 특정 단어가 몇번 나오는가

IDF(Inverse Document Frequency) : 역문서 빈도
log(N/(t가 등장한 문서의 개수)) (N : 전체 문서수)
전체 컬렉션에서 해당 단어가 얼마나 희귀한가
"""

'\nTF-IDF : 단어 중요도를 숫자로 표현\n\nTF(Term Frequency) : 용어 빈도\n해당 문서에서 특정 단어가 몇번 나오는가\n\nIDF(Inverse Document Frequency) : 역문서 빈도\n전체 컬렉션에서 해당 단어가 얼마나 희귀한가\n'

In [47]:
import math

class TFIDF:
    def __init__(self, document):
        self.docs = document
        self.tokenized = [doc.lower().split() for doc in documents]
        self.N = len(documents)
        self.df = {}
        for tokens in self.tokenized:
            for t in set(tokens):
                self.df[t] = self.df.get(t, 0) + 1

    def tf(self, term, doc_tokens):
        return doc_tokens.count(term) / len(doc_tokens)

    def idf(self, term):
        return math.log(self.N / self.df.get(term, 1))

    def score(self, query, doc_idx):
        tokens = self.tokenized[doc_idx]
        return sum(self.tf(t, tokens) * self.idf(t) for t in query.lower().split())

    def search(self, query, top_k=3):
        scores = [(i, self.score(query, i)) for i in range(self.N)]
        return sorted(scores, key=lambda x:x[1], reverse=True)[:top_k]

In [48]:
tfidf = TFIDF(documents)
for idx, score in tfidf.search('Python 프로그래밍'):
    print(f"[{idx}] {score}")

[0] 0.28782313662425574
[1] 0.0
[2] 0.0


In [74]:
from collections import Counter

class BM25:
    def __init__(self, documents, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs = documents
        self.tokenized = [doc.lower().split() for doc in documents]
        self.N = len(documents)
        self.avgdl = sum(len(d) for d in self.tokenized) / self .N
        self.df = {}
        for tokens in self.tokenized:
            for t in set(tokens):
                self.df[t] = self.df.get(t, 0) + 1

    def idf(self, term):
        df = self.df.get(term, 0)
        return math.log((self.N -df + 0.5) / (df + 0.5) + 1)

    def score(self, query, doc_idx):
        tokens = self.tokenized[doc_idx]
        dl = len(tokens)
        tf_counter = Counter(tokens)
        total = 0.0
        for t in query.lower().split():
            tf = tf_counter.get(t, 0)
            numerator = tf * (self.k1 + 1)
            denominator = tf + (self.k1 * (1-self.b + self.b * dl / self.avgdl))
            total += self.idf(t) * numerator / denominator
        return total

    def search(self, query, top_k=3):
        scores = [(i, self.score(query, i)) for i in range(self.N)]
        return sorted(scores, key=lambda x:x[1], reverse=True)[:top_k]

In [75]:
bm25 = BM25(documents)
for idx, score in bm25.search("python 프로그래밍 언어"):
    print(f"Document {idx}: {documents[idx]}")
    print(f"Score: {score:.3f}")

Document 0: Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니다.
Score: 2.036
Document 3: GPT-4는 OpenAI가 개발한 대규모 언어 모델로 다양한 작업을 수행합니다.
Score: 1.930
Document 1: 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.
Score: 0.000


In [62]:
from langchain_community.retrievers import BM25Retriever

In [38]:
from langchain_core.documents import Document

In [41]:
docs_lc = [Document(page_content = d, metadata={'index':i}) for i, d in enumerate(documents)]
bm25_retriever = BM25Retriever.from_documents(docs_lc)

In [40]:
!uv add rank_bm25

Resolved 224 packages in 479ms                                       
Prepared 1 package in 46ms                                               
Installed 1 package in 3ms                                  
 + rank-bm25==0.2.2


In [42]:
bm25_retriever.k = 3

In [43]:
results = bm25_retriever.invoke('Python 프로그래밍')
results

[Document(metadata={'index': 0}, page_content='Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니다.'),
 Document(metadata={'index': 9}, page_content='임베딩은 텍스트를 수치 벡터로 변환하여 의미적 유사성을 측정할 수 있게 합니다.'),
 Document(metadata={'index': 8}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 입력을 설계하는 기술입니다.')]

In [51]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(docs_lc, embeddings)
vector_retriever = vectorstore.as_retriever(search_kwarg = {'k' : 3})

In [52]:
result = vector_retriever.invoke('딥러닝 모델 구조')
result

[Document(id='87993d50-729e-45ed-84b3-42ef7000d697', metadata={'index': 6}, page_content='트랜스포머 아키텍처는 어텐션 메커니즘을 활용한 딥러닝 모델 구조입니다.'),
 Document(id='b68613d9-4832-44d1-8055-10aab4bd1eaf', metadata={'index': 3}, page_content='GPT-4는 OpenAI가 개발한 대규모 언어 모델로 다양한 작업을 수행합니다.'),
 Document(id='84d64623-bf04-4ba7-bf81-7bead401114f', metadata={'index': 4}, page_content='RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM 응답을 개선합니다.'),
 Document(id='c311cb2d-3aa4-4688-ac59-be1d5b7499f9', metadata={'index': 0}, page_content='Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니다.')]

In [53]:
from langchain_classic.retrievers import EnsembleRetriever

In [64]:
ensemble = EnsembleRetriever(
    retrievers = [vector_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

In [65]:
results = ensemble.invoke('Python 데이터 과학')
results

[Document(id='c311cb2d-3aa4-4688-ac59-be1d5b7499f9', metadata={'index': 0}, page_content='Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니다.'),
 Document(id='159f66d0-7b24-4fec-ad64-9b86b397a71b', metadata={'index': 9}, page_content='임베딩은 텍스트를 수치 벡터로 변환하여 의미적 유사성을 측정할 수 있게 합니다.'),
 Document(id='f9ec0c44-b0c9-4d57-98c0-33f40ad6cdc1', metadata={'index': 2}, page_content='벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하는 시스템입니다.'),
 Document(metadata={'index': 8}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 입력을 설계하는 기술입니다.'),
 Document(id='74751b4e-ccc2-4753-bbda-2a09ea03d576', metadata={'index': 1}, page_content='자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.')]

In [67]:
for w_vec, w_bm25 in [(0.2, 0.8), (0.5, 0.5), (0.8, 0.2)]:
    ensemble = EnsembleRetriever(
        retrievers = [vector_retriever, bm25_retriever],
        weights=[w_vec, w_bm25]
        )
    results = ensemble.invoke('벡터 데이터베이스 검색')
    top_idx = results[0].metadata['index']
    print(f"BM25 = {w_bm25}, Vector= {w_vec} -> Top-1 : {top_idx} {results[0].page_content[:30]}")

BM25 = 0.8, Vector= 0.2 -> Top-1 : 7 FAISS는 Facebook AI가 개발한 효율적인 유
BM25 = 0.5, Vector= 0.5 -> Top-1 : 2 벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 
BM25 = 0.2, Vector= 0.8 -> Top-1 : 2 벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 
